# Import Required Libraries
Import necessary libraries such as `matplotlib`, `serial`, and `time` for plotting, serial communication, and delays.

In [3]:
# Import necessary libraries
import matplotlib.pyplot as plt  # For plotting
import serial  # For serial communication
import time  # For delays and timing
import re  # For parsing the received data
import numpy as np  # For numerical operations

In [ ]:
# Define a function to initialize serial communication with the ESP32
def initialize_serial(port, baudrate=115200, timeout=1):
    """
    Initializes the serial communication with the ESP32.
    
    Parameters:
        port (str): The COM port to which the ESP32 is connected.
        baudrate (int): The baud rate for serial communication. Default is 115200.
        timeout (int): Timeout for serial communication in seconds. Default is 1.
    
    Returns:
        serial.Serial: The initialized serial object.
    """
    try:
        ser = serial.Serial(port, baudrate, timeout=timeout)
        print(f"Serial connection established on port {port} at {baudrate} baud.")
        return ser
    except Exception as e:
        print(f"Error initializing serial communication: {e}")
        return None

# Define a function to send a command to the ESP32
def send_command(ser, command):
    """
    Sends a command to the ESP32 via serial communication.
    
    Parameters:
        ser (serial.Serial): The serial object for communication.
        command (str): The command string to send.
    """
    if ser and ser.is_open:
        ser.write((command + '\n').encode())  # Send the command with a newline character
        print(f"Command sent: {command}")
    else:
        print("Serial connection is not open. Unable to send command.")

# Define a function to read data from the ESP32
def read_data(ser):
    """
    Reads data from the ESP32 via serial communication.
    
    Parameters:
        ser (serial.Serial): The serial object for communication.
    
    Returns:
        str: The data received from the ESP32.
    """
    if ser and ser.is_open:
        try:
            data = ser.readline().decode().strip()  # Read a line of data and decode it
            return data
        except Exception as e:
            print(f"Error reading data: {e}")
            return None
    else:
        print("Serial connection is not open. Unable to read data.")
        return None

# Define a function to collect acceleration and RPM data
def collect_acceleration_and_rpm_data(ser, percentages):
    """
    Collects acceleration (X, Y, Z) and RPM data from the ESP32 for a list of percentage values.
    
    Parameters:
        ser (serial.Serial): The serial object for communication.
        percentages (list): A list of percentage values to send.
    
    Returns:
        tuple: Two lists - one with RPM values and the other with acceleration data (X, Y, Z).
    """
    rpms = []
    accel_x = []
    accel_y = []
    accel_z = []

    for percentage in percentages:
        # Send the 'start <percentage>' command
        command = f"start {percentage}"
        send_command(ser, command)
        time.sleep(0.5)  # Wait for the ESP32 to process the command and send data

        while True:
            data = read_data(ser)
            if data:
                # Parse acceleration data
                if data.startswith("X:"):
                    try:
                        # Extract acceleration values using regex
                        match = re.search(r"X:\s*(-?\d+\.?\d*)\s+Y:\s*(-?\d+\.?\d*)\s+Z:\s*(-?\d+\.?\d*)", data)
                        if match:
                            accel_x.append(float(match.group(1)))
                            accel_y.append(float(match.group(2)))
                            accel_z.append(float(match.group(3)))
                            print(f"Acceleration Data - X: {match.group(1)}, Y: {match.group(2)}, Z: {match.group(3)}")
                    except Exception as e:
                        print(f"Error parsing acceleration data: {e}")
                
                # Parse RPM data
                elif data.startswith("RPM is:"):
                    try:
                        rpm = float(data.split(":")[1].strip())
                        rpms.append(rpm)
                        print(f"RPM Data: {rpm}")
                        break  # Exit the loop after collecting RPM data
                    except ValueError:
                        print(f"Invalid RPM data received: {data}")
            else:
                print("No data received. Retrying...")
    
    return rpms, accel_x, accel_y, accel_z

# Example usage
# ser = initialize_serial('/dev/ttyUSB0')  # Replace with your ESP32's serial port
# percentages = [0, 20, 40, 60, 80, 100]
# rpms, accel_x, accel_y, accel_z = collect_acceleration_and_rpm_data(ser, percentages)

# Plot the collected data
def plot_acceleration_vs_rpm(rpms, accel_x, accel_y, accel_z):
    """
    Plots the acceleration values (X, Y, Z) against the corresponding RPM values.
    
    Parameters:
        rpms (list): A list of RPM values.
        accel_x (list): A list of acceleration values for the X-axis.
        accel_y (list): A list of acceleration values for the Y-axis.
        accel_z (list): A list of acceleration values for the Z-axis.
    """
    plt.figure(figsize=(12, 8))  # Set the figure size
    
    # Plot acceleration for X-axis
    plt.plot(rpms, accel_x, marker='o', linestyle='-', color='r', label='Acceleration X vs RPM')
    
    # Plot acceleration for Y-axis
    plt.plot(rpms, accel_y, marker='o', linestyle='-', color='g', label='Acceleration Y vs RPM')
    
    # Plot acceleration for Z-axis
    plt.plot(rpms, accel_z, marker='o', linestyle='-', color='b', label='Acceleration Z vs RPM')
    
    plt.title('Acceleration vs RPM Plot')  # Set the title of the plot
    plt.xlabel('RPM')  # Label for the x-axis
    plt.ylabel('Acceleration (m/s²)')  # Label for the y-axis
    plt.grid(True)  # Enable grid for better readability
    plt.legend()  # Add a legend
    plt.show()  # Display the plot

# Example usage for plotting
# plot_acceleration_vs_rpm(rpms, accel_x, accel_y, accel_z)

In [ ]:
ser = initialize_serial('/dev/ttyUSB0')  # Replace with your ESP32's serial port
percentages = numpy.linspace(0, 10, num=4)
rpms, accel_x, accel_y, accel_z = collect_acceleration_and_rpm_data(ser, percentages)
plot_acceleration_vs_rpm(rpms, accel_x, accel_y, accel_z)